# Combined Cycle Power Plant（联合循环电厂工况）


In [ ]:
from pathlib import Path

# 工具入参要求绝对路径；本 cell 算好后经 {VAR} 注入下方各命令
HERE = Path.cwd()
PY = str(HERE / "power_plant.py")      # :load / :load_X / :load_X_to_predict 按需拼后缀
YAML = str(HERE / "power_plant.yaml")
# 注意：power_plant.py 读同目录 power_plant.csv（相对路径），本 notebook 须在本目录下逐 cell 运行


In [ ]:
# ① 数据画像（power_plant.py:load 全量数据）
!uv --project ../.. run --extra data mflowy data_profile '{"file_path": "{PY}:load"}'


In [ ]:
# ② 探索性数据分析（power_plant.py:load 全量数据）
!uv --project ../.. run --extra data mflowy eda '{"file_path": "{PY}:load", "target": "net_hourly_electrical_energy_output"}'

In [ ]:
# ③ 多模型对比（从输出中记录 XGB 步骤的 run_id）
!uv --project ../.. run --extra model --extra deep mflowy modeling '{"modeling_steps_yaml": "{YAML}", "name": "power-v1", "desc": "Power Plant modeling"}'


In [ ]:
# ④ SHAP 解释（把下方 aedc2efee19444ba829e36159977d557 替换为 ③ 输出的裸 run_id（无 XGB= 前缀））
!uv --project ../.. run --extra model --extra deep mflowy explanation '{"modeling_steps_yaml": "{YAML}", "model": "aedc2efee19444ba829e36159977d557", "name": "power-shap", "desc": "Power SHAP"}'


In [ ]:
# ⑤ 逆向搜索：目标出力最大化（power_plant.py:load_X 提供搜索空间，替代 X.csv）
!uv --project ../.. run --extra model --extra deep mflowy inverse_optimization '{"data": "{PY}:load_X", "model": "aedc2efee19444ba829e36159977d557", "direction": {"net_hourly_electrical_energy_output": "maximize"}, "n_trials": 300}'


In [ ]:
# ⑥ 上线前预测验证（power_plant.py:load_X_to_predict 提供推荐工况单行，替代 recommended.csv）
!uv --project ../.. run --extra model --extra deep mflowy predict '{"data": "{PY}:load_X_to_predict", "model": "aedc2efee19444ba829e36159977d557"}'
